## **<mark>Import the required packages and functions</mark>**

In [72]:
from pyspark.sql.functions import to_timestamp,col,when, from_utc_timestamp,expr,\
window, row_number, current_timestamp, lit, max
from pyspark.sql.types import StringType, IntegerType, DoubleType,TimestampType
from pyspark.sql import Row
from pyspark.sql.window import Window
import uuid
from datetime import datetime
from geopy.geocoders import Nominatim

StatementMeta(, 1c262e2a-410d-4eb3-a89f-be736ba3246a, 76, Finished, Available, Finished, False)

## **<mark>Creating Audit variables</mark>**

In [82]:
run_id = str(uuid.uuid4())
pipeline_run_id = ""      # Pipeline parameter
notebook_name = "Silver Notebook"
start_time = datetime.now()
status = "none"
records_processed = 0
error_message = "none"

StatementMeta(, 1c262e2a-410d-4eb3-a89f-be736ba3246a, 86, Finished, Available, Finished, False)

## **<mark>Silver Layer ETL Code inside Try, Except, Finally for Notebook Audit purpose</mark>**

In [84]:
try:
    #=====================================================================================================
    # Incremental load logic
    #=====================================================================================================

    # here we take only the new rows for incremental load using the "silver layer max ingection time processed" 

    if spark.catalog.tableExists("LH_Weather_Update.silver.weather_selected"):
    
        silver_last_processed_time = (spark.sql("""SELECT MAX(ingestion_time) AS max_time
            FROM LH_Weather_Update.silver.weather_selected""").first()["max_time"])

        df = spark.sql(f"""SELECT *FROM LH_Weather_Update.bronze.weather_data
            WHERE ingestion_time > TIMESTAMP('{silver_last_processed_time}')""")
    
    
    else:
        
        df = spark.sql("""SELECT *FROM LH_Weather_Update.bronze.weather_data""")

    #=================
    # Type casting 
    #=================

    df = df.withColumn("Datetime", to_timestamp("Datetime")) \
       .withColumn("utc_offset_seconds",col("utc_offset_seconds").cast(IntegerType()))
    
    #=================
    # Null handaling
    #=================

    # here in this data set "Datetime" and "Temperature" is critical measures,
    # if any row get null that row become not mesurable we have to ignore that rows.

    qc_df = df.withColumn("qc_result",when(col("Datetime").isNull(), "Missing Datetime")
            .when(col("Temperature").isNull(), "Missing Temperature"))

    # here we spliting "good" and "bad" records using isnull() and isnotnull() function.

    good_df = qc_df.filter(col("qc_result").isNull())
    bad_df = qc_df.filter(col("qc_result").isNotNull()) 

    #=============================
    # Removing Dupicates
    #=============================
    # Metadata columns

    meta_cols = ["ingestion_time", "source_file"]

    # Business columns
    business_cols = [c for c in df.columns 
                        if c not in meta_cols]

    # Remove duplicates based on business columns only

    good_df = good_df.dropDuplicates(subset= business_cols)

    bad_df = bad_df.dropDuplicates(subset=business_cols)

    # always when you have to take a deduplication method only you have to consider a business columns,
    # you have to ignore the meta columns. 

    # make a qc_result of good_rows dataframe as "passed"
    good_df = good_df.withColumn("qc_result", lit("passed"))

    # creating a date column in indian datetime format (+5.30)
    good_df = good_df.withColumn("Datetime_IST",from_utc_timestamp("Datetime", "Asia/Kolkata"))

    #=============================
    # QC Audit table creation
    #=============================

    # data for Data quality audit table
    # if using count always use cache it stores the result in RAM, if you use count next it take data fron RAM no need to read all the rows

    total_counts = df.cache().count()
    good_counts = good_df.cache().count()
    bad_counts = bad_df.cache().count()
    if total_counts > 0:
        quality_score = round((good_counts / total_counts) * 100, 2)
    else:
        quality_score = 0.0

    duplicate_count = df.count() - df.dropDuplicates(subset=business_cols).count()

    df = df.dropDuplicates(subset=business_cols)

    dq_audit_df = spark.createDataFrame([(
    str(uuid.uuid4()),
    pipeline_run_id,
    "Silver",
    total_counts,
    good_counts,
    bad_counts,
    duplicate_count,
    quality_score,
    datetime.now())],
        ["RunID",
        "PipelineRunID",
        "LayerName",
        "TotalRecords",
        "GoodRecords",
        "BadRecords",
        "DuplicateRecords",
        "QualityScore",
        "CreatedOn"])

    # save the QC audit table
    dq_audit_df.write.mode("append").saveAsTable("audit.data_qc_table")

    #=============================
    # creating reverce Gro-Decoder
    #=============================

    # Initialize geolocator
    geolocator = Nominatim(user_agent="fabric_weather_project")

    # Reverse geocoder function (no try/except)
    def get_location(lat, lon):
        try:
            location = geolocator.reverse(
            f"{lat},{lon}",
            language="en")
            address = location.raw.get("address", {})

            return (
            address.get("city")
            or address.get("town")
            or address.get("village"),
            address.get("state"),
            address.get("country")
        )

        except Exception:

            return (
            "Unknown",
            "Unknown",
            "Unknown"
            )

    # Extract distinct coordinates
    distinct_locations = df.select("latitude", "longitude").distinct()

    # Reverse Geocode Only Unique Locations
    location_rows = []
    for row in distinct_locations.collect():
        lat = row["latitude"]
        lon = row["longitude"]

        city, state, country = get_location(lat, lon)
        location_rows.append((lat, lon, city, state, country))

    # Create Location Lookup DataFrame
    location_df = spark.createDataFrame(
        location_rows,["latitude", "longitude", "city", "state", "country"])

    # Join Back to Weather Data
    good_df = good_df.join(location_df, ["latitude", "longitude"], "inner")

    #==========================================
    # Saving the good and bad rows to lakehouse
    #==========================================

    spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
    good_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("silver.weather_selected")
    bad_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("silver.weather_regected")

    records_processed = total_counts

    status = "Success"

except Exception as e:

    status = "Failed"
    error_message = str(e)

    raise

finally:

    end_time = datetime.now()

    duration = (end_time - start_time).total_seconds()

    notebook_audit_df = spark.createDataFrame(
        [(
            str(uuid.uuid4()),
            pipeline_run_id,
            notebook_name,
            status,
            records_processed,
            start_time,
            end_time,
            duration,
            error_message,
            datetime.now()
        )],
        [
            "RunID",
            "PipelineRunID",
            "NotebookName",
            "Status",
            "RecordsProcessed",
            "StartTime",
            "EndTime",
            "DurationSeconds",
            "ErrorMessage",
            "CreatedOn"])

    notebook_audit_df.write.mode("append").saveAsTable("audit.notebook_audit")

StatementMeta(, 1c262e2a-410d-4eb3-a89f-be736ba3246a, 88, Finished, Available, Finished, False)